# Solutions — Effects

Only look here after you've actually tried the exercises in `useeffect.ipynb`.

### LESSON 39 — Exercise

**1. Two render lines, one effect line, per press.**

```text
   render — count is 1, screen shows "0"
   render — count is 1, screen shows "0"
   effect — count is 1, screen shows "1"  <- already updated
```

The renders are doubled by `<StrictMode>` in development (LESSON 3): React calls your
component an extra time to expose impure code. The **commit** is not doubled, and the Effect
runs at the end of a commit — so one press, one Effect.

The sentence to take away: React may call your component more times than you expect, and that
is deliberately safe *because* rendering is supposed to change nothing. The Effect is where
changing something is allowed, and it runs once.

**2. Setting the same value: renders, but no Effect.**

The step that did not happen is **commit**.

`setCount(count)` hands React a value that is `Object.is`-equal to the current one. React
still calls the component — that is why the render lines appear — but the result is identical
to what is already on screen, so there is nothing to apply and no commit happens. Effects run
at the end of a *commit*. No commit, no Effect.

This is the cleanest demonstration in the whole topic of why LESSON 38 separated those three
words. "It re-rendered" and "it committed" are different events, and the Effect is attached to
the second one.

**3. `document.title` in the Effect.** A legitimate use, because the document title is
**outside React**. React owns what it renders inside `#root`; it does not own the browser tab,
the page title, the URL, or anything else on the page. Writing to the title is exactly the
phrase from the lesson — *synchronize your component with some system outside of React* — and
the thing that causes it is the component being on screen showing this count, not any
particular click.

**4. The same line in the handler.** It works, and for this experiment it is honestly fine —
which is why the question is worth asking rather than assuming the Effect is automatically
better.

The version to keep is the **Effect**, and the reason is in the question: the handler version
only runs when *that button* is clicked. The moment the count can change any other way — a
second button, a reset, a value arriving from a prop, the component mounting with a count that
is not zero — the title silently stops matching. The Effect version is tied to the *state*,
not to one route by which the state can change, so it cannot fall out of step.

That is the general rule and it is worth keeping: put it in the handler if it is caused by
**that interaction**; put it in an Effect if it must be true **whenever the component is on
screen in this state**.

**Common mistakes.**

- Reading the doubled render lines as a bug. It is `<StrictMode>` and LESSON 3 warned about it.
- Expecting an Effect after the same-value click. No commit, no Effect.
- Concluding from question 4 that handlers are wrong and Effects are right. Most side effects
  in a real app belong in handlers — LESSON 44 makes that case properly.
- Reading the DOM during render because the experiment does. The experiment says in a comment
  that this is instrumentation and impure; it is there to prove the timing, not as a pattern.

### LESSON 39 — Mini challenge

| | belongs in | why |
|---|---|---|
| 1. analytics ping on Send | **event handler** | caused by one specific click |
| 2. chat server connection while the screen shows | **Effect** | caused by the component being on screen, however the user got there |
| 3. basket total under the list | **neither** | it is a derived value — calculate it during render (LESSON 29) |
| 4. `document.title` for the current product | **Effect** | the title must match whatever is displayed, not one click; and the title is outside React |
| 5. countdown timer when a quiz screen appears | **Effect** | caused by the screen appearing; a timer is outside React, and it will need cleanup (LESSON 41) |
| 6. validation message after submit | **event handler** | caused by the submit — this is exactly LESSON 32-33 |
| 7. filtering a list by a search box, to display it | **neither** | derived from state during render (LESSON 21, LESSON 29) |

**The two that are the same trap: 3 and 7.**

Both are values you can calculate from what you already hold, and both are tempting to write
as an Effect that computes something and stores it with a setter:

```jsx
// the trap
useEffect(() => {
  setTotal(items.reduce((sum, item) => sum + item.price, 0));
}, [items]);
```

(The `[items]` at the end is the *dependency array* — LESSON 40. Ignore it for now; it is
there because real code has it, and it is not the part that is wrong. The bug is the setter.)

That is LESSON 29's duplicated-state bug wearing an Effect as a disguise. It costs an extra
render every time, it can show a stale total for one frame, and it has to be kept in step by
hand — all to avoid one line above the `return`:

```jsx
const total = items.reduce((sum, item) => sum + item.price, 0);
```

If your Effect's whole job is to set state from other state, the Effect should not exist.
LESSON 44 is that lesson; noticing it here is the point of the challenge.

### LESSON 40 — Exercise

**Part 1.**

In [ ]:
function l40changed(previous, next) {
  if (previous === null || previous === undefined) return true;   // first render
  return next.some((dep, i) => !Object.is(dep, previous[i]));
}

const l40pairs = [
  ['["general", 3] vs ["general", 3]', ["general", 3], ["general", 3]],
  ['["general", 3] vs ["random", 3] ', ["general", 3], ["random", 3]],
  ['[{id:1}]       vs [{id:1}]      ', [{ id: 1 }], [{ id: 1 }]],
  ['[]             vs []            ', [], []],
];

for (const [label, prev, next] of l40pairs) {
  console.log(label, "-> re-runs?", l40changed(prev, next));
}

console.log("first render          -> re-runs?", l40changed(null, ["general"]));

// 3. The third pair looks unchanged to a human because the CONTENTS match. React never
//    compares contents - Object.is compares identity, and those are two separate objects
//    built at two different moments, so it concludes the dependency changed and re-runs.
//
//    The fourth pair is the empty array: there is nothing to differ, so `some` is false and
//    the Effect never re-runs after the first time. That is what `[]` means.

Note what `l40changed` is and is not. It is the documented comparison rule applied to two
ordinary arrays — nothing here models how React stores or schedules anything.

**Part 2 — the playground.**

**1. Clicking `random` twice.** The Effect re-runs **once**, on the first click.

The first click changes `roomId` from `"general"` to `"random"`, so the dependency differs and
React cleans up the old connection and sets up a new one. The second click sets `roomId` to
`"random"` when it is already `"random"`; `Object.is("random", "random")` is `true`, the
dependency has not changed, and the Effect is left alone. Measured:

```text
cleanup — disconnected from "general"
setup   — connected to "random"
(second click: nothing)
```

**2. With `[]`.** The setup runs once and never again. Clicking the room buttons still updates
the screen — `roomId` is state, so the component re-renders and the `<b>` shows the new room —
but the console says the connection is still attached to whichever room was current at mount.

That gap is the point: **the UI and the outside world have drifted apart.** The screen claims
you are in `random` while the connection is still in `general`. An empty dependency array is a
promise that the Effect depends on nothing, and here that promise is false.

**3. Adding an object literal as a dependency.** The Effect now cleans up and re-runs on
*every* commit, including clicks that change nothing about the room — because `options` is
built fresh each render and `Object.is` sees a new value every time.

The line that predicted it: *"An object, array or function created during render is a new
value every render, so listing it as a dependency means the Effect never stops re-running."*

**Common mistakes.**

- Reading `[]` as "run once" and stopping there. It also means "this Effect depends on
  nothing", which is a claim about your code that the linter — and reality — can contradict.
- Adding items to the dependency array until the warning goes away. The list is not a dial;
  it is a description of what the Effect reads.
- Removing a dependency to stop an Effect looping. That hides the loop rather than fixing it,
  and the Effect then runs with stale values.
- Assuming two objects with the same fields are the same dependency. They never are.

### LESSON 40 — Mini challenge

**1. `Profile` — should be `[userId]`.** As written, `[]` means the Effect runs once and never
again, so navigating from one profile to another keeps loading the *first* user. The screen
would show a new name while the loaded data stays stale — the same UI/outside-world drift as
Part 2.2.

**2. `Search` — should be `[query, pageSize]`.** `pageSize` is read inside the Effect and is
missing. Change the page size and the search silently keeps using the old one. This is the
most common real bug of the four, because it works perfectly until someone touches the
forgotten value.

**3. `Timer` — should be `[seconds]`.** `config` is an object built during render, so it is a
new value on every render and the Effect restarts the timer after every commit. Depend on the
primitive the object was built from.

**4. `Title` — `[fullName]` is correct as written.**

**Why 4 is fine and 3 is not.** Both list a value created in the component body, but
`fullName` is a **string** and `config` is an **object**. `Object.is("Ada Lovelace", "Ada
Lovelace")` is `true` — a primitive built from the same inputs is the same value, every time.
`Object.is({seconds: 30}, {seconds: 30})` is `false` — a fresh object is a different value
however identical its contents.

The idea underneath is LESSON 27's: objects are compared by identity, not by contents. It
decided whether React re-renders (LESSON 25), whether a state update was a mutation
(LESSON 27-28), and now whether an Effect re-runs. Same rule, third appearance.

### LESSON 41 — Exercise

In [ ]:
// 1. The first three lines on load:
//
//      setup   — connected to "general"      <- development stress-test
//      cleanup — disconnected from "general" <- development stress-test
//      setup   — connected to "general"      <- the real one
//
//    Strict Mode runs one extra setup+cleanup cycle BEFORE the first real setup. The third
//    line is the connection that actually survives; the first two exist only to check that
//    the cleanup undoes the setup.
//
// 2. Switching general -> random prints:
//
//      cleanup — disconnected from "general"
//      setup   — connected to "random"
//
//    The cleanup names the OLD room because it was created during the render where roomId
//    was "general", and it closed over that value. React runs it "with the old values" on
//    purpose: undoing something requires knowing what was done.
//
// 3. Unmounting prints cleanup alone; mounting again prints the full setup/cleanup/setup
//    sequence. Unmounting only has to stop; mounting has to start, and in development it is
//    stress-tested first.
//
// 4. Deleting the cleanup: every room switch now opens a connection and never closes it, so
//    after five switches there are five live connections and only the last one is doing
//    anything useful. Nothing is visible because the experiment's "connection" is a
//    console.log - there is no resource to exhaust. In a real app you would see memory grow,
//    duplicate messages arriving on every stale socket, and handlers firing several times per
//    event. This is exactly the failure Strict Mode's double-run is designed to surface.
//
// 5. The cleanup logs the label built during the render that created it - so switching from
//    general to random logs "GENERAL". A cleanup function sees the values from its OWN setup
//    render, not the current ones. That is what makes it able to undo the right thing.

console.log("setup starts synchronising; cleanup stops it - with the values it started with");

**Common mistakes.**

- Returning something other than a cleanup function from an Effect. React expects a function
  or nothing; an `async` Effect returns a Promise, which is why LESSON 43 has to deal with it.
- Reading the doubled mount lines as a double connection in production. It is development
  only.
- Silencing the double-run by removing `<StrictMode>`. That removes the test, not the bug.
- Expecting the cleanup to see current values. It sees the values from the render that made
  it, and that is the feature.

### LESSON 41 — Mini challenge

| | the cleanup should |
|---|---|
| 1. `setInterval` | `clearInterval(id)` — otherwise a second timer starts on every re-run and they all keep firing |
| 2. `addEventListener` | `window.removeEventListener("resize", handleResize)` with the same function reference |
| 3. socket + listener | close the socket, and remove the `message` listener — setup did two things, so cleanup undoes both |
| 4. `document.title` | **nothing — no cleanup needed** |

**Why 4 is different.** The first three *start* something that keeps running after the Effect
finishes: a repeating timer, a registered listener, an open connection. Each one accumulates if
you never stop it.

Setting `document.title` starts nothing. It is a one-off assignment that completes immediately
and leaves nothing behind to stop; the next Effect simply overwrites it. There is no "undo"
that would make sense — restoring the previous title would be undoing the wrong thing, since
the point was to change it.

The test to apply: **after this Effect's body finishes, is anything still running because of
it?** If yes, cleanup. If no, none needed.

**What the colleague chose not to find out.** In number 1, `setInterval` with no cleanup means
every re-run adds another timer. With `<StrictMode>` on, the extra setup+cleanup cycle makes
that visible immediately in development: two timers, `tick()` firing twice a second instead of
once. Commenting out `<StrictMode>` makes the symptom disappear in development — and leaves it
intact in production, where it appears as a counter that speeds up the longer the page is
open, or a clock that jumps two seconds at a time. They have not fixed the duplicate timer;
they have arranged not to see it until a user does.

### LESSON 42 — Exercise

**1. Fast versus slow typing.** Fast, five letters:

```text
scheduled — "r" · cancelled — "r" · scheduled — "re" · cancelled — "re" · … · SEARCHING — "react"
```

Four `cancelled` lines and exactly **one** `SEARCHING`. Typing slowly with a pause after each
letter produces a `SEARCHING` line for every letter, because each timer was allowed to finish
before the next keystroke changed `query`.

The one sentence: **debouncing does not reduce the number of keystrokes, it reduces the number
that survive long enough to matter** — and a pause longer than the delay means they all do.

**2. `DELAY = 2000`.** Typing and waiting works but feels broken: two seconds of nothing after
you stop. Typing continuously for ten seconds produces **no search at all** — every keystroke
cancels the previous timer, and the timer never gets its full two seconds.

That is the real lesson about delay: a debounce that is too long does not merely feel slow, it
can mean the work never happens while the user is active. The delay must be shorter than a
natural pause in the activity you are debouncing.

**3. Deleting the cleanup.** Every keystroke gets its own surviving timer, so a five-letter
word runs **five** searches, 500ms after each keystroke:

```text
scheduled — "r" · scheduled — "re" · scheduled — "rea" · … then five SEARCHING lines
```

What stopped happening is LESSON 41's guarantee: React no longer has anything to run before
re-running the Effect, so nothing cancels the pending timer. The debounce disappears entirely
and you are back to one request per keystroke — with the extra insult that the results arrive
in whatever order the timers finish.

**4. `return clearTimeout(id);`** would call `clearTimeout` **immediately**, during the Effect,
and return its result — which is `undefined`. So the timer is cancelled the instant it is
created and the search never runs at all; and React gets `undefined` where it expected a
cleanup function, so there is no cleanup either.

The rule it violates is LESSON 41's: an Effect returns a **function** to be called later, not
the result of calling something now. It is LESSON 22's `onClick={fn()}` mistake in a different
costume.

**Common mistakes.**

- Debouncing the state update itself so the input becomes laggy. Keep the controlled input
  instant (LESSON 30) and debounce only the expensive consequence.
- Putting the timer in an event handler instead of an Effect, then having nothing to cancel it
  when the component disappears mid-timer.
- Assuming a debounce guarantees one request. It guarantees one request *per pause*. A user who
  types steadily for a minute triggers none; a user who pauses five times triggers five.
- Reaching for a debounce library. It is six lines and you have just written them.

### LESSON 42 — Mini challenge

| | debounce? | why |
|---|---|---|
| 1. search-as-you-type | **yes** | fewer network requests, the canonical case |
| 2. 5000 rows re-rendering slowly | **no** | the work happens *during rendering* — topic 23 |
| 3. auto-saving a draft | **yes** | fewer writes, and a pause is exactly the right trigger |
| 4. "load more" button | **no** | one deliberate click, one request; nothing to collapse |
| 5. live collaborative typing | **no** | the keystrokes *are* the payload — delaying them defeats the feature |

**The two about rendering: 2 and — partly — 5.** Number 2 is the clear one. The list is slow
because React is rendering 5000 rows; debouncing the filter does not make that render any
faster, it just makes it happen less often, so the interface goes from *always janky* to
*janky in bursts, after a delay*. React's own documentation makes this exact point: debouncing
and throttling are "blocking… they merely postpone the moment when rendering blocks the
keystroke". There is a purpose-built answer for render work and it is topic 23's.

Number 5 fails for a different reason worth separating: the work is outside React, so
debouncing *would* function — but the feature is "see my collaborator typing live", and a
debounce deletes the feature. The tool fits and the requirement does not.

### LESSON 43 — Exercise

In [ ]:
// 1. With the ignore flag on, typing "a" then quickly "ab":
//
//      request  — "a"          <- one-letter query, deliberately slow (1500ms)
//      request  — "ab"         <- longer query, fast (300ms)
//      applied  — results for "ab"
//      ignored  — "a" (a newer request is in flight)
//
//    The "a" request was still in flight when "ab" finished. It arrived roughly 1.2 seconds
//    later and was discarded.
//
// 2. With the flag off, the fourth line becomes `applied — results for "a"` and the result
//    line shows results for "a" while the search box still reads "ab".
//
//    As a user would report it: "I searched for something, and it showed me results for what
//    I typed a second ago." Or more often: "the results are wrong sometimes." There is no
//    error, nothing in the console, and it only happens when they type quickly.
//
// 3. `let ignore = false` is declared INSIDE the Effect, so a new one is created every time
//    the Effect runs. Each run's async function closes over its own copy, and each run's
//    cleanup sets its own copy. That is the same closure behaviour LESSON 41 relied on when
//    the cleanup remembered the OLD room - a function created during a render keeps the
//    values from that render.
//
//    If `ignore` lived outside the component there would be one shared variable, and the
//    first cleanup would silence every request including the current one.
//
// 4. With every query taking the same 300ms, the responses come back in the order they were
//    sent, and the bug does not appear - with or without the flag.
//
//    That is exactly why race conditions are hard to catch: on a fast, consistent network
//    they never happen. They appear on a slow connection, on a bad mobile signal, when one
//    endpoint is briefly slower than another, or when a user types faster than usual. The
//    code is equally broken in all cases; only the timing differs.

console.log("last response wins - and it may not be the one you want");

**Common mistakes.**

- `useEffect(async () => …)`. The commonest of all; React gets a Promise instead of a cleanup.
- Declaring `ignore` outside the Effect, or in a module. One shared flag silences everything.
- Checking the flag before the `await` instead of after. The point is to check it once the
  response has come back, not before it was ever requested.
- Trying to "fix" the doubled development request with a guard. It is Strict Mode, the flag
  already makes it harmless, and production sends one request.

### LESSON 43 — Mini challenge

**1. A and B are broken. C is correct.**

**A** makes the Effect callback `async`, so it returns a Promise. React expects a cleanup
function or nothing, so there is no cleanup at all — and therefore no way to ignore a stale
response. Two faults from one keyword.

**B** is the interesting one.

**2. Why B fails.** It declares `ignore`, it sets `ignore = true` in the cleanup — and then
never reads it. The `.then` callback calls `setUser(user)` unconditionally. The flag is
bookkeeping that nothing consults, so every response is applied, stale ones included.

This is worth dwelling on because it is what a half-remembered fix looks like: all the parts
are present and one `if` is missing. The flag only does something at the moment you check it:

```js
getUser(userId).then((user) => { if (!ignore) setUser(user); });
```

**3. "Just cancel the request."** React's sentence is the honest answer:

> You can't "undo" a network request that already happened, but your cleanup function should
> ensure that the fetch that's *not relevant anymore* does not keep affecting your application.

The server has already been asked, and in the general case it has already done the work. What
you control is your own reaction to the answer. The `ignore` flag protects **your state**, not
your bandwidth — it guarantees that an obsolete response cannot change what the user sees.

(There is a real cancellation mechanism for `fetch`, and it saves bandwidth rather than
correctness. Even with it you still want the flag, because a response can arrive in the gap
between deciding to cancel and the cancellation taking effect.)

### LESSON 44 — Exercise

**Part 1 — three Effects that should not exist.**

In [ ]:
// A - a string built from two others
function l44fullName(first, last) {
  return `${first} ${last}`;
}

// B - a filtered view of a list
function l44visible(items, category) {
  return items.filter((item) => item.category === category);
}

// C - a boolean derived from a length
function l44isEmpty(items) {
  return items.length === 0;
}

const l44items = [
  { id: 1, label: "Bread", category: "food" },
  { id: 2, label: "Soap", category: "home" },
  { id: 3, label: "Milk", category: "food" },
];

console.log("A:", l44fullName("Ada", "Lovelace"));
console.log("B:", JSON.stringify(l44visible(l44items, "food").map((i) => i.label)));
console.log("C:", l44isEmpty(l44items), "|", l44isEmpty([]));

// Each replacement is one expression above the return. None needs state, none needs an
// Effect, none can go stale, and none costs a second render pass.

**Part 2 — judgement.**

In [ ]:
// 1. modal opened by a button ............ HANDLER   caused by that click
// 2. page became visible, however reached . EFFECT    caused by being on screen, not by one route
// 3. character count under a textarea ..... NEITHER   derived from the text (LESSON 29)
// 4. save a draft 500ms after typing stops  EFFECT    debounced, outside React (LESSON 42)
// 5. "mark as read" when a message clicked  HANDLER   caused by that click
// 6. websocket while a screen is open ..... EFFECT    the canonical case - connect/disconnect
// 7. sorting a table on a header click .... NEITHER   sort during render from a sort-key state
//
// 7 is worth a second look. The CLICK is an event, and the handler's job is to record which
// column was chosen - that is one setState. The sorted rows themselves are derived from the
// data plus that choice, so they are calculated during render (LESSON 21). Storing a sorted
// copy in state would be LESSON 29's duplication all over again.

console.log("what caused it, and is anything outside React involved");

**Common mistakes.**

- Replacing an Effect with a `useMemo` by reflex. Most derived values need neither; `useMemo`
  is topic 23 and needs a measurement first.
- Treating "no Effect" as "no state". Number 7 still needs state — for the *choice*, not for
  the sorted result.
- Moving an Effect into a handler when the thing really is caused by rendering. Number 2 and
  number 6 belong in Effects, and pushing them into handlers means missing every other route
  the user could take.

### LESSON 44 — Mini challenge

**1. Is the double-run the bug?** No. In this lesson's words: *"if remounting breaks the logic
of your application, this usually uncovers existing bugs."* Strict Mode did not invent a second
purchase; it demonstrated that the code charges on *appearing*, and appearing can happen more
than once. Removing Strict Mode removes the demonstration.

**2. A production sequence with no Strict Mode involved.** The user buys the product, then
presses **Back**, then **Forward** — the product page mounts again, the Effect runs again, they
are charged again. Or: they open the page, wander off to another route, come back to compare,
and are charged a second time without clicking anything. Or the page is in a tab they reopen
from history. None of these involve development mode, and all of them are things users do
constantly.

**3. Where it belongs, and the question that places it.** In the Buy button's click handler:

```jsx
function handleClick() {
  fetch("/api/buy", { method: "POST", body: JSON.stringify({ productId }) });
}
```

The question is LESSON 39's: **what caused this?** Buying is caused by a person deciding to
buy and pressing a button — one specific interaction. It is not caused by the product page
being on screen. Asked once, the code could not have been written any other way.

**4. Why the `hasBought` guard is worse.** It papers over the symptom and adds a new bug.

It does silence the double charge, so it looks like a fix. But `hasBought` is a module-level
variable: it belongs to the whole application, not to a purchase, and it is never reset. The
user's **second genuine purchase** — of the same product, or of anything else if the guard is
shared — silently does nothing. They click Buy, nothing happens, no error appears, and they
try again. You have converted an over-charging bug into an under-charging bug and made it
harder to find, because now the behaviour depends on what the user did earlier in the session.

The code still charges on render. The guard only limits how often.